In [1]:
import numpy as np
import os
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['Times New Roman'] + plt.rcParams['font.serif']

## Load and Display Chest X-ray Dataset

In [ ]:
IMG_SIZE = 64  # resize all images to 64x64 pixels

def load_images_from_folder(folder, label, img_size=IMG_SIZE):
    images = []
    labels = []
    for filename in sorted(os.listdir(folder)):
        if not filename.lower().endswith(('.jpeg', '.jpg', '.png')):
            continue
        filepath = os.path.join(folder, filename)
        img = Image.open(filepath).convert('L')  # convert to grayscale
        img = img.resize((img_size, img_size))
        img_array = np.array(img).flatten() / 255.0  # flatten and normalize to [0,1]
        images.append(img_array)
        labels.append(label)
    return images, labels

data_dir = 'dataset/chest_xray'

# Load training data (NORMAL = 0, PNEUMONIA = 1)
train_normal_imgs, train_normal_labels = load_images_from_folder(
    os.path.join(data_dir, 'train', 'NORMAL'), label=0)
train_pneumonia_imgs, train_pneumonia_labels = load_images_from_folder(
    os.path.join(data_dir, 'train', 'PNEUMONIA'), label=1)

X_train = np.array(train_normal_imgs + train_pneumonia_imgs)
y_train = np.array(train_normal_labels + train_pneumonia_labels).reshape(-1, 1)

# Load test data
test_normal_imgs, test_normal_labels = load_images_from_folder(
    os.path.join(data_dir, 'test', 'NORMAL'), label=0)
test_pneumonia_imgs, test_pneumonia_labels = load_images_from_folder(
    os.path.join(data_dir, 'test', 'PNEUMONIA'), label=1)

X_test = np.array(test_normal_imgs + test_pneumonia_imgs)
y_test = np.array(test_normal_labels + test_pneumonia_labels).reshape(-1, 1)

print('X_train.shape', X_train.shape)
print('X_test.shape', X_test.shape)
print('y_train.shape', y_train.shape)
print('y_test.shape', y_test.shape)

In [ ]:
# Check class distribution
print('Training set:')
print('  NORMAL:', np.sum(y_train == 0))
print('  PNEUMONIA:', np.sum(y_train == 1))
print('Test set:')
print('  NORMAL:', np.sum(y_test == 0))
print('  PNEUMONIA:', np.sum(y_test == 1))

In [ ]:
# Plot some sample images
ncols = 5
nrows = 2
fig, ax = plt.subplots(nrows=nrows, ncols=ncols, figsize=[15, 6])
class_names = ['NORMAL', 'PNEUMONIA']
for i in range(nrows):
    idx = np.where(y_train.flatten() == i)[0]
    samples = np.random.choice(idx, ncols, replace=False)
    for j in range(ncols):
        ax[i, j].imshow(X_train[samples[j]].reshape(IMG_SIZE, IMG_SIZE), cmap='gray')
        ax[i, j].axis('off')
        if j == 0:
            ax[i, j].set_ylabel(class_names[i], fontsize=14)
fig.suptitle('Sample Chest X-ray Images', fontsize=16)
plt.tight_layout()

## Dimensionality Reduction with PCA

Since the input dimensionality is high (64 x 64 = 4096 features per image), we use PCA to reduce the feature space before training.

In [ ]:
n_components = 50
pca = PCA(n_components=n_components)
X_train_pca = pca.fit_transform(X_train)
X_test_pca = pca.transform(X_test)

print('X_train_pca.shape', X_train_pca.shape)
print('X_test_pca.shape', X_test_pca.shape)
print('Cumulative explained variance:', np.round(np.sum(pca.explained_variance_ratio_), 4))

# Plot cumulative explained variance
fig, ax = plt.subplots(figsize=[8, 4])
ax.plot(np.arange(1, n_components + 1), np.cumsum(pca.explained_variance_ratio_), 'o-')
ax.set_xlabel('Number of Components', fontsize=12)
ax.set_ylabel('Cumulative Explained Variance', fontsize=12)
ax.set_title('PCA Explained Variance', fontsize=14)
ax.grid(True)
plt.tight_layout()

## Logistic Regression

Binary Logistic Regression:  \sim \text{Bernoulli}(Q)$,  = \sigma(H^T W)$ where $\sigma(x) = e^x/(1+e^x)$.

MLE: Find $\hat{W} = \arg\min_W \sum_j \left( \log(1+e^{H_j^T W}) - Y^T H^T W \right)$

In [ ]:
# sigmoid function
def sigmoid(x):
    return np.exp(x) / (1 + np.exp(x))

In [ ]:
def fit_LR_GD(Y, H, W0=None, sub_iter=100, stopping_diff=0.01):
    '''
    Convex optimization for Logistic Regression using Gradient Descent
    Y = (n x 1), H = (p x n) (Phi in lecture notes), W = (p x 1)
    Logistic Regression: Y ~ Bernoulli(Q), Q = sigmoid(H.T @ W)
    MLE -->
    Find W_hat = argmin_W ( sum_j ( log(1+exp(H_j.T @ W) ) - Y.T @ H.T @ W ) )
    '''
    if W0 is None:
        W0 = np.random.rand(H.shape[0], 1)

    W1 = W0.copy()
    i = 0
    grad = np.ones(W0.shape)
    while (i < sub_iter) and (np.linalg.norm(grad) > stopping_diff):
        Q = 1 / (1 + np.exp(-H.T @ W1))  # probability matrix, same shape as Y
        grad = H @ (Q - Y)
        W1 = W1 - (np.log(i + 1) / (((i + 1) ** (0.5)))) * grad
        i = i + 1
        if i % 50 == 0 or i == 1:
            print('iter %i, grad_norm %f' % (i, np.linalg.norm(grad)))
    return W1

In [ ]:
# Feature matrix H of size (p x n) = (feature dim + 1) x (num samples)
# Add first row of 1's for bias features
H_train = np.vstack((np.ones(X_train_pca.shape[0]), X_train_pca.T))
print('H_train.shape', H_train.shape)

# Fit logistic regression using gradient descent
W = fit_LR_GD(Y=y_train, H=H_train, sub_iter=300)
print('W.shape', W.shape)

In [ ]:
# Project regression coefficients back to pixel space for visualization
W_pixel = pca.components_.T @ W[1:]  # (4096, 50) @ (50, 1) = (4096, 1)

fig, ax = plt.subplots(nrows=1, ncols=1, figsize=[5, 5])
im = ax.imshow(W_pixel.reshape(IMG_SIZE, IMG_SIZE), cmap='seismic')
ax.set_title('Logistic Regression Coefficients (pixel space)', fontsize=13)
fig.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()

In [ ]:
# Get predicted probabilities on test set
H_test = np.vstack((np.ones(X_test_pca.shape[0]), X_test_pca.T))
Q = 1 / (1 + np.exp(-H_test.T @ W))  # predicted probabilities

# Convert probabilities to binary predictions (threshold = 0.5)
Y_pred = (Q >= 0.5).astype(int)

print('Sample predictions (first 10):')
print('  Predicted probs: ', np.round(Q[:10].flatten(), 3))
print('  Predicted labels:', Y_pred[:10].flatten())
print('  True labels:     ', y_test[:10].flatten())

## Evaluation

In [ ]:
# TODO: Compute accuracy = (TP + TN) / (TP + TN + FP + FN)
# TODO: Compute sensitivity (recall) = TP / (TP + FN)
# TODO: Compute specificity = TN / (TN + FP)
# TODO: Compute precision = TP / (TP + FP)
# TODO: Compute F1-score = 2 * (precision * recall) / (precision + recall)
# TODO: Plot confusion matrix
# TODO: Compute AUC-ROC and plot ROC curve
# TODO: K-fold cross-validation
# TODO: Plot learning curves (training vs validation loss)